# Perturbation Performance vs. Number of Perturbed Genes

Three messages in one plot:
1. Using **more genes** to perturb neighbours improves performance
2. As performance improves, we **approach counterfactual** (graph-swap) quality
3. **Cell-type-specific** logFC outperforms global logFC at every gene count

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('src')

import numpy as np
import pandas as pd
import decoupler as dc
import matplotlib.pyplot as plt

from cellina import CellinaModel, make_neighbor_perturbation
from src.cellina._spatial_utils import spatial_neighbors, compute_spatial_features
from perturb_utils import load_crc_slide, compute_cf_logfc

plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
slide_id             = 242
labels_key           = 'coarse_type'
domains_key          = 'typ'
top_n                = 100                        # metric evaluation window (fixed)
top_n_perturb_values = [5, 20, 50, 100, 200, 500] # sweep
batch_size           = 512
min_cells            = 50

ref_label = f'{slide_id}_REF'
crc_label = f'{slide_id}_CRC'

## 1. Data

In [ ]:
adata = load_crc_slide(slide_id, labels_key=labels_key, domains_key=domains_key)
print(adata)

In [ ]:
spatial_neighbors(adata, bandwidth=np.inf, max_neighbours=50, standardize=False)
compute_spatial_features(adata)

## 2. Model

In [ ]:
CellinaModel.setup_anndata(
    adata,
    batch_key=None,
    labels_key=labels_key,
    domains_key=domains_key,
    layer='counts',
    spatial_obsm_key='spatial_x',
)

model = CellinaModel(
    adata,
    n_latent=20,
    classifier_lambda=1,
    discriminator_lambda=1,
    n_layers=3,
    condition_on_intrinsic=False,
)

model.train(
    max_epochs=100,
    check_val_every_n_epoch=1,
    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_monitor='vae_loss_validation',
    train_size=0.9,
    validation_size=0.1,
    plan_kwargs={'lr': 0.001, 'weight_decay': 0.0001, 'normalize_losses': True},
    enable_checkpointing=True,
    batch_size=batch_size,
    devices=[0],
)

## 3. Pseudobulk logFC — Global & Cell-type-specific

In [ ]:
# Global: all cells pooled together
pdata_global = dc.pp.pseudobulk(
    adata=adata, sample_col=domains_key, groups_col=None, mode='sum', layer='counts'
)
sc.pp.normalize_total(pdata_global, target_sum=1e4)
sc.pp.log1p(pdata_global)

global_logfc_series = pd.Series(
    (pdata_global[pdata_global.obs[domains_key] == crc_label].X
     - pdata_global[pdata_global.obs[domains_key] == ref_label].X).flatten(),
    index=pdata_global.var_names,
)

# Cell-type-specific
pdata_ct = dc.pp.pseudobulk(
    adata=adata, sample_col=domains_key, groups_col=labels_key, mode='sum', layer='counts'
)
sc.pp.normalize_total(pdata_ct, target_sum=1e4)
sc.pp.log1p(pdata_ct)

cell_types_with_both = [
    ct for ct in pdata_ct.obs[labels_key].unique()
    if ((pdata_ct.obs[domains_key] == ref_label) & (pdata_ct.obs[labels_key] == ct)).any()
    and ((pdata_ct.obs[domains_key] == crc_label) & (pdata_ct.obs[labels_key] == ct)).any()
]

domain_logfc_df = pd.concat(
    [
        pd.Series(
            (pdata_ct[(pdata_ct.obs[domains_key] == crc_label) & (pdata_ct.obs[labels_key] == ct)].X
             - pdata_ct[(pdata_ct.obs[domains_key] == ref_label) & (pdata_ct.obs[labels_key] == ct)].X
            ).flatten(),
            index=pdata_ct.var_names,
            name=ct,
        )
        for ct in cell_types_with_both
    ],
    axis=1,
).T

print(f"Global logFC: {global_logfc_series.shape[0]} genes")
print(f"CT-specific logFC: {domain_logfc_df.shape} (cell types × genes)")

## 4. Pre-compute Fixed Expressions per Cell Type

In [ ]:
import scanpy as sc  # needed for sc.pp calls above in case not imported

ref_idxs = {}   # ct → indices in REF
crc_idxs = {}   # ct → indices in CRC
ref_exprs = {}  # ct → (n_ref, n_genes)
cf_exprs  = {}  # ct → (n_crc, n_genes)  real CRC, ground truth
swap_exprs = {} # ct → (n_ref, n_genes)  counterfactual (graph swap)

cell_types = []
for ct in sorted(cell_types_with_both):
    ref_idx = np.where(
        (adata.obs[labels_key] == ct) & (adata.obs[domains_key] == ref_label)
    )[0]
    crc_idx = np.where(
        (adata.obs[labels_key] == ct) & (adata.obs[domains_key] == crc_label)
    )[0]
    if len(ref_idx) < min_cells or len(crc_idx) < min_cells:
        print(f"  skip {ct}: ref={len(ref_idx)}, crc={len(crc_idx)}")
        continue
    print(f"  {ct}: ref={len(ref_idx)}, crc={len(crc_idx)}")
    ref_idxs[ct]  = ref_idx
    crc_idxs[ct]  = crc_idx
    ref_exprs[ct]  = model.get_normalized_expression(indices=ref_idx, batch_size=batch_size, library_size='latent')
    cf_exprs[ct]   = model.get_normalized_expression(indices=crc_idx, batch_size=batch_size, library_size='latent')
    swap_exprs[ct] = model.get_counterfactual_expression(ref_idx, crc_idx, batch_size=batch_size)
    cell_types.append(ct)

print(f"\nEvaluating {len(cell_types)} cell types: {cell_types}")

## 5. Counterfactual Baseline (Fixed)

In [ ]:
cf_pearson_vals = []
for ct in cell_types:
    stats = compute_cf_logfc(ref_exprs[ct], swap_exprs[ct], cf_exprs[ct], top_n=top_n)
    cf_pearson_vals.append(stats['pearson_r'])

cf_avg_pearson = float(np.mean(cf_pearson_vals))
print(f"Counterfactual avg Pearson r = {cf_avg_pearson:.3f}")

## 6. Sweep: Global vs Cell-type-specific

In [ ]:
global_results = []
ctspec_results = []

for n in top_n_perturb_values:
    print(f"\n── top_n_perturb = {n} ──")

    # ── Global perturbation ──────────────────────────────────────────────────
    top_genes = global_logfc_series.abs().nlargest(n).index.tolist()
    logfc_dict = {g: float(global_logfc_series[g]) for g in top_genes}
    make_neighbor_perturbation(adata, perturbations=logfc_dict, obsm_key_out='spatial_x_cf', base=np.e)

    pearson_vals = []
    for ct in cell_types:
        pert_expr = model.get_perturbed_expression(
            adata=adata, indices=ref_idxs[ct], spatial_obsm_key='spatial_x_cf',
            batch_size=batch_size, library_size='latent',
        )
        stats = compute_cf_logfc(ref_exprs[ct], pert_expr, cf_exprs[ct], top_n=top_n)
        pearson_vals.append(stats['pearson_r'])
    avg_g = float(np.mean(pearson_vals))
    global_results.append(avg_g)
    print(f"  global:      Pearson r = {avg_g:.3f}")

    # ── Cell-type-specific perturbation ──────────────────────────────────────
    logfc_series_dict = {}
    for ct in domain_logfc_df.index:
        s = domain_logfc_df.loc[ct]
        top_g = s.abs().nlargest(n).index.tolist()
        logfc_series_dict[ct] = s[top_g]
    make_neighbor_perturbation(
        adata, perturbations=logfc_series_dict, groupby=labels_key,
        obsm_key_out='spatial_x_cf', base=np.e,
    )

    pearson_vals = []
    for ct in cell_types:
        pert_expr = model.get_perturbed_expression(
            adata=adata, indices=ref_idxs[ct], spatial_obsm_key='spatial_x_cf',
            batch_size=batch_size, library_size='latent',
        )
        stats = compute_cf_logfc(ref_exprs[ct], pert_expr, cf_exprs[ct], top_n=top_n)
        pearson_vals.append(stats['pearson_r'])
    avg_c = float(np.mean(pearson_vals))
    ctspec_results.append(avg_c)
    print(f"  CT-specific: Pearson r = {avg_c:.3f}")

# clean up temporary obsm key
if 'spatial_x_cf' in adata.obsm:
    del adata.obsm['spatial_x_cf']

## 7. Summary Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    top_n_perturb_values, global_results,
    color='#4C72B0', ls='--', marker='o', ms=6, lw=1.8,
    label='Global logFC',
)
ax.plot(
    top_n_perturb_values, ctspec_results,
    color='#DD8452', ls='-', marker='s', ms=6, lw=1.8,
    label='Cell-type-specific logFC',
)
ax.axhline(
    cf_avg_pearson,
    color='#2ca02c', ls=':', lw=1.8,
    label=f'Counterfactual / graph swap  (r = {cf_avg_pearson:.3f})',
)

ax.set_xscale('log')
ax.set_xticks(top_n_perturb_values)
ax.set_xticklabels([str(v) for v in top_n_perturb_values])
ax.set_xlabel('N perturbed genes per cell type', fontsize=12)
ax.set_ylabel(f'Avg. Pearson r  (top-{top_n} genes vs real CRC)', fontsize=12)
ax.set_title('Perturbation performance vs. number of perturbed genes', fontsize=12)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.show()